# Capítulo 6 — Modelos de Linguagem de Grande Escala em Inteligência e OSINT

**Inteligência Artificial e Machine Learning Avançado para Defesa** — Prof. Cristiano Alves · Quantum Strategic AI

---

### 🎯 Objetivos do capítulo

Ao final deste notebook, você será capaz de:

1. **Explicar** a arquitetura *Transformer* — **auto-atenção**, atenção de **múltiplas cabeças** e **codificação posicional** — e por que ela substituiu a recorrência;
2. **Distinguir** as duas grandes famílias de modelos de linguagem — baseados em **codificador** (BERT, para *compreensão*) e em **decodificador** (GPT, para *geração*) — e os seus usos adequados;
3. **Aplicar** os dois paradigmas de adaptação — ***fine-tuning*** e ***prompt engineering*** / aprendizado em contexto — e a **geração ancorada em recuperação (RAG)**, conhecendo suas trocas;
4. **Empregar** modelos de linguagem em OSINT: análise automatizada, classificação, sumarização, síntese de relatórios e apoio à decisão;
5. **Reconhecer e mitigar** os riscos próprios desses modelos — **alucinação**, **vazamento de dados** e **dependência de modelos estrangeiros** — como preocupações operacionais de primeira ordem.

> O Capítulo 5 terminou com uma provocação: o mecanismo de atenção resolvera o gargalo da tradução, e talvez a recorrência fosse dispensável. Este capítulo leva essa tese à sua conclusão — a arquitetura ***Transformer*** — e, sobre ela, os **modelos de linguagem de grande escala** (*Large Language Models*, LLMs) que redefiniram o processamento de linguagem na última década. Com eles, fechamos o Módulo II.
>
> É o capítulo **mais consequente da obra até aqui, e o mais ambivalente**. Os LLMs oferecem, em inteligência, um salto de capacidade — leem e sintetizam volumes de texto em escala inacessível a qualquer equipe humana. Mas trazem riscos que, em defesa, não são detalhes técnicos: um modelo que **inventa fatos com fluência**, que **expõe dados sensíveis a terceiros**, ou de cuja **tecnologia estrangeira se depende**, apresenta vulnerabilidades operacionais e estratégicas de primeira ordem. Por isso a disciplina da obra — comparar, validar, manter o humano na decisão — aqui deixa de ser boa prática e passa a ser **condição de emprego responsável**.

## Preparação do ambiente

O coração do capítulo — a **auto-atenção** da Listagem 6.1 e a codificação posicional — executa **offline, em `numpy` puro**. As Seções 6.2 a 6.6 usam modelos pré-treinados via `transformers` e `sentence-transformers`, instalados em célula própria; os pesos (~2 GB no total: BERT multilíngue, codificador de recuperação e um modelo gerador) são baixados na primeira execução. No Colab, uma GPU acelera a inferência, mas **tudo executa também em CPU**.

In [ ]:
# Preparacao do ambiente: importacoes e semente de reprodutibilidade
import numpy as np
import matplotlib
import matplotlib.pyplot as plt

SEMENTE = 0
np.random.seed(SEMENTE)

print(f"numpy      {np.__version__}")
print(f"matplotlib {matplotlib.__version__}")
print("Ambiente pronto.")

---
## 6.1 A arquitetura Transformer

A atenção do Capítulo 5 permitia ao decodificador consultar o codificador. O **Transformer** (Vaswani et al., 2017, no artigo de título sugestivo *Attention Is All You Need*) generaliza a ideia e **elimina a recorrência**: cada elemento da sequência atende a todos os demais **da mesma sequência** — a **auto-atenção** (*self-attention*).

### 6.1.1 Auto-atenção de produto escalar

O mecanismo é elegante. De cada vetor de entrada derivam-se, por projeções aprendidas, três vetores: uma **consulta** (*query*), uma **chave** (*key*) e um **valor** (*value*). A saída de cada posição é uma soma dos valores de todas as posições, ponderada pela compatibilidade entre a sua consulta e as chaves das demais. Em forma matricial, empilhando consultas, chaves e valores em $Q$, $K$ e $V$,

$$\mathrm{Atenção}(Q, K, V) = \mathrm{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right) V,$$

onde $d_k$ é a dimensão das chaves e o fator $1/\sqrt{d_k}$ estabiliza os escores antes do *softmax*. O produto $QK^\top$ mede a **afinidade entre cada par de posições**; o *softmax* a transforma em pesos; a multiplicação por $V$ produz, para cada posição, uma representação que **mistura informação de toda a sequência**. É assim que uma palavra passa a ter um **vetor contextual** — que depende das palavras ao redor —, superando a limitação das *embeddings* estáticas do Capítulo 5.

A **Listagem 6.1** implementa o mecanismo em poucas linhas:

In [ ]:
# Listagem 6.1 - Auto-atencao de produto escalar escalonado, em NumPy.
def softmax(x, eixo=-1):
    x = x - x.max(axis=eixo, keepdims=True)   # estabilidade numerica
    e = np.exp(x)
    return e / e.sum(axis=eixo, keepdims=True)

def auto_atencao(X, Wq, Wk, Wv):
    # X: (n_tokens, d_modelo). Wq, Wk, Wv: projecoes (d_modelo, d_k).
    Q = X @ Wq                                # consultas
    K = X @ Wk                                # chaves
    V = X @ Wv                                # valores

    d_k = Q.shape[-1]
    escores = Q @ K.T / np.sqrt(d_k)          # afinidade consulta-chave
    pesos = softmax(escores, eixo=-1)         # normaliza por linha
    saida = pesos @ V                         # soma ponderada dos valores
    return saida, pesos

# A saida de cada token e uma mistura dos valores de TODOS os tokens,
# ponderada pela compatibilidade: eis a representacao contextual.
print("Funcoes softmax() e auto_atencao() definidas.")

In [ ]:
# A auto-atencao em acao sobre uma frase com token repetido
# (projecoes aleatorias com semente: a MECANICA e a mesma da rede
# treinada; os pesos reais seriam aprendidos por retropropagacao)
rng = np.random.default_rng(SEMENTE)

frase = "o comboio cruzou a ponte e o comboio seguiu ao norte".split()
vocab = sorted(set(frase))
d_modelo, d_k = 16, 8

# embedding por TIPO de token: ocorrencias da mesma palavra recebem
# o mesmo vetor de entrada (como em uma camada de embedding real)
emb = {t: rng.normal(0, 1, d_modelo) for t in vocab}
X = np.vstack([emb[t] for t in frase])

Wq = rng.normal(0, 0.5, (d_modelo, d_k))
Wk = rng.normal(0, 0.5, (d_modelo, d_k))
Wv = rng.normal(0, 0.5, (d_modelo, d_k))

saida, pesos = auto_atencao(X, Wq, Wk, Wv)
print(f"entrada X: {X.shape} -> saida contextual: {saida.shape}")
print(f"cada linha da matriz de pesos soma {pesos.sum(axis=1).round(6)[0]}")

plt.figure(figsize=(7.5, 6))
plt.imshow(pesos, cmap="Blues")
plt.colorbar(label="peso de atencao")
plt.xticks(range(len(frase)), frase, rotation=45, ha="right")
plt.yticks(range(len(frase)), frase)
plt.xlabel("atende a (chaves)")
plt.ylabel("posicao (consultas)")
plt.title("Auto-atencao: cada token atende a todos os demais")
plt.tight_layout(); plt.show()

**Observe:** cada linha da matriz soma **1** (é um *softmax* por consulta — o Exercício 6.1 verifica isso e o papel do $1/\sqrt{d_k}$). Note as duas ocorrências de *comboio*: por terem o **mesmo vetor de entrada**, produzem consultas e chaves idênticas — e se atendem mutuamente com o mesmo peso. Em um modelo **treinado**, é a atenção que faz cada ocorrência absorver o seu **contexto** (a ponte, o norte) e sair com vetores *diferentes* — contextuais.

### 6.1.2 Múltiplas cabeças, posição e a pilha completa

Três acréscimos completam a arquitetura:

- A **atenção de múltiplas cabeças** executa várias atenções em paralelo, cada uma em um subespaço projetado distinto, para captar relações de tipos diferentes (sintáticas, semânticas, de correferência), e depois as concatena;
- A **codificação posicional** injeta a *ordem* das palavras — necessária porque a auto-atenção, sem recorrência, é **indiferente à permutação** —, tipicamente por padrões senoidais ou vetores aprendidos;
- Cada bloco combina atenção e uma rede *feed-forward* com **conexões residuais** e **normalização de camada** (Capítulo 3), empilhados dezenas de vezes.

A razão do triunfo do Transformer é prática tanto quanto conceitual: sem a dependência sequencial da RNN, ele **paraleliza o treino**, o que permitiu escalá-lo a volumes de dados e de computação antes impensáveis — e é dessa **escala**, mais do que de qualquer engenhosidade adicional, que emergem as capacidades dos grandes modelos de linguagem.

> **📝 Nota** — A escala transformou uma diferença de grau em uma **diferença de tipo**. Treinados sobre corpora massivos, os modelos maiores exibem capacidades — seguir instruções, aprender com exemplos no próprio *prompt*, raciocinar por etapas — que os menores não possuem, sem que nenhuma delas tenha sido programada explicitamente. Esse **comportamento emergente** é o que torna os LLMs tão poderosos e, ao mesmo tempo, tão **difíceis de prever e auditar** — uma tensão que percorrerá o restante deste capítulo.

In [ ]:
# Codificacao posicional senoidal: a "regua" que informa a ordem
def codificacao_posicional(n_posicoes, d_modelo):
    PE = np.zeros((n_posicoes, d_modelo))
    pos = np.arange(n_posicoes)[:, None]
    i = np.arange(0, d_modelo, 2)[None, :]
    freq = 1.0 / (10000 ** (i / d_modelo))
    PE[:, 0::2] = np.sin(pos * freq)
    PE[:, 1::2] = np.cos(pos * freq)
    return PE

PE = codificacao_posicional(n_posicoes=60, d_modelo=64)

plt.figure(figsize=(9, 4.6))
plt.imshow(PE, aspect="auto", cmap="RdBu_r")
plt.colorbar(label="valor")
plt.xlabel("dimensao do vetor")
plt.ylabel("posicao na sequencia")
plt.title("Codificacao posicional senoidal: cada posicao recebe uma assinatura unica")
plt.tight_layout(); plt.show()

# Sem ela, "o drone abateu o alvo" e "o alvo abateu o drone" seriam
# INDISTINGUIVEIS para a auto-atencao (indiferenca a permutacao).
print("Assinaturas de posicoes vizinhas sao parecidas; distantes, nao:")
print(f"  cosseno(pos 10, pos 11) = "
      f"{np.dot(PE[10], PE[11]) / (np.linalg.norm(PE[10]) * np.linalg.norm(PE[11])):.2f}")
print(f"  cosseno(pos 10, pos 50) = "
      f"{np.dot(PE[10], PE[50]) / (np.linalg.norm(PE[10]) * np.linalg.norm(PE[50])):.2f}")

**Observe:** a codificação dá a cada posição uma **assinatura única e suave** — posições próximas têm assinaturas parecidas, o que permite à rede raciocinar sobre distância e ordem sem recorrência.

---
## 6.2 Duas famílias: BERT e GPT

O Transformer original tinha um codificador e um decodificador. Duas famílias de modelos **especializaram cada metade**, com vocações distintas.

O **BERT** (*Bidirectional Encoder Representations from Transformers*) usa a pilha **codificadora**. É **bidirecional** — cada palavra enxerga o contexto à esquerda e à direita — e treina-se prevendo palavras mascaradas no texto (*masked language modeling*). Produz ***embeddings* contextuais** de alta qualidade e brilha em tarefas de **compreensão**: classificação, extração de entidades, recuperação de informação.

O **GPT** (*Generative Pre-trained Transformer*) usa a pilha **decodificadora**. É **autorregressivo**: prevê a próxima palavra dada a sequência anterior, modelando $P(x_t \mid x_1, \ldots, x_{t-1})$, e gera texto amostrando essa distribuição repetidamente. Brilha em tarefas de **geração**: redação, sumarização, resposta a perguntas — e é a base dos modelos de instrução e de diálogo.

A distinção orienta a escolha da ferramenta: modelos da família **BERT para classificar e extrair** (mais baratos e fáceis de implantar); da família **GPT para sintetizar e redigir**.

A **Listagem 6.2** usa um BERT pré-treinado para gerar as *embeddings* contextuais que resolvem, de vez, a ambiguidade de *manga* do Capítulo 5. Antes, a célula seguinte instala as bibliotecas do capítulo (série 4.x da `transformers`, como no Capítulo 5):

In [ ]:
# Bibliotecas das Secoes 6.2-6.6 (no Colab, ~1 min; pesos baixados
# na primeira execucao de cada modelo)
%pip install -q "transformers<5" sentencepiece sentence-transformers

In [ ]:
# Listagem 6.2 - Embeddings contextuais com um BERT pre-treinado.
import os
os.environ["USE_TF"] = "0"     # backend PyTorch (evita conflito com Keras 3)
from transformers import AutoTokenizer, AutoModel
import torch

# BERT multilingue pre-treinado: gera embeddings CONTEXTUAIS.
nome = "bert-base-multilingual-cased"
tok = AutoTokenizer.from_pretrained(nome)
bert = AutoModel.from_pretrained(nome)

def embeddings_contextuais(frase):
    entrada = tok(frase, return_tensors="pt")
    with torch.no_grad():
        saida = bert(**entrada)
    # um vetor por token, da ultima camada oculta
    return saida.last_hidden_state[0], tok.tokenize(frase)

# A palavra "manga" recebe vetores DIFERENTES em cada contexto -- o que
# as embeddings estaticas do Capitulo 5 nao conseguiam distinguir.
v_fruta, tokens_fruta = embeddings_contextuais("comi uma manga madura")
v_roupa, tokens_roupa = embeddings_contextuais("rasguei a manga da camisa")
print("tokens:", tokens_fruta, "|", tokens_roupa)

In [ ]:
# O mesmo token, vetores diferentes: a ambiguidade de "manga" resolvida
def vetor_do_token(vetores, tokens, alvo="manga"):
    # +1 compensa o token especial [CLS] no inicio da sequencia
    return vetores[tokens.index(alvo) + 1]

def cosseno(a, b):
    a, b = a.numpy(), b.numpy()
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))

m_fruta = vetor_do_token(v_fruta, tokens_fruta)
m_roupa = vetor_do_token(v_roupa, tokens_roupa)

print("similaridade de cosseno entre os vetores de 'manga':")
print(f"  fruta ~ roupa (acepcoes DIFERENTES): {cosseno(m_fruta, m_roupa):.3f}")

# com embeddings ESTATICAS (Cap. 5), essa similaridade seria 1.000 --
# um unico vetor por palavra, qualquer que fosse o contexto.

**Observe:** a similaridade entre as duas *mangas* é **claramente menor que 1** — cada ocorrência carrega o seu contexto. Com as *embeddings* estáticas do Capítulo 5, seria exatamente 1,000 (o Exercício 6.2 completa a análise comparando dois usos na *mesma* acepção).

---
## 6.3 Adaptação: fine-tuning, prompt engineering e RAG

Um LLM pré-treinado é **generalista**. Especializá-lo a uma tarefa operacional admite três caminhos, com trocas distintas:

- O ***fine-tuning*** continua o treino do modelo sobre dados rotulados da tarefa, ajustando seus pesos. Entrega a **melhor acurácia** para uma tarefa fixa e produz um modelo que **se possui e controla** — vantagem de soberania —, ao custo de dados rotulados e computação. Variantes eficientes (**LoRA**, adaptadores) reduzem muito esse custo, ajustando apenas uma pequena fração dos parâmetros.

- O ***prompt engineering***, ou **aprendizado em contexto**, orienta um modelo *congelado* apenas pela formulação do *prompt* — instruções e, quando útil, exemplos (*few-shot*). Não exige treino e é extremamente flexível, mas é **menos confiável** e, no caso de modelos acessados por API de terceiros, **envia os dados para fora**.

- O terceiro caminho, hoje central em aplicações sérias, é a **geração ancorada em recuperação** (*Retrieval-Augmented Generation*, **RAG**): em vez de confiar no que o modelo "sabe", **recupera-se** de uma base de documentos o material relevante à pergunta e fornece-se ao modelo **como contexto**, pedindo que responda a partir dele e **cite as fontes**. O RAG ataca de frente os dois maiores problemas do emprego operacional: **reduz a alucinação** (a resposta ancora-se em fontes reais) e adiciona **auditabilidade** (as fontes são citadas e verificáveis), sem retreinar o modelo e permitindo injetar conhecimento proprietário e atual. É o padrão que estrutura a aplicação integrada deste capítulo (Seção 6.6).

> **🛡️ Contexto de defesa** — As três estratégias têm implicações diretas de **segurança e soberania**. O *prompt engineering* sobre uma **API estrangeira** é o caminho mais rápido e o mais arriscado: expõe os dados e depende de um serviço fora de controle nacional. O ***fine-tuning* de um modelo aberto**, executado em infraestrutura própria, é mais custoso, porém preserva o sigilo e a soberania sobre a capacidade. O **RAG sobre um modelo auto-hospedado** combina o melhor dos mundos: conhecimento atual e citável, sem enviar dados sensíveis a terceiros. A escolha, em defesa, não é apenas técnica — é uma decisão de **opsec e de soberania tecnológica** (Capítulo 1).

---
## 6.4 LLMs em OSINT: análise, sumarização e apoio à decisão

O ganho operacional dos LLMs em inteligência é de **síntese**. A triagem dos Capítulos 4 e 5 *priorizava* a atenção humana; os LLMs vão além e **condensam** o material priorizado — leem coleções de documentos, extraem entidades e relações, classificam com poucos exemplos, resumem grandes volumes e redigem rascunhos de relatórios de situação. Em OSINT, onde o volume de fontes abertas é esmagador, essa capacidade é um multiplicador de força que **transforma o analista de leitor em revisor**.

A **sumarização** ilustra bem o ganho *e* o risco. A **Listagem 6.3** produz um resumo **abstrativo** de um texto longo. Mas o resumo *reescreve* o original e pode introduzir afirmações que nele não constavam — uma forma de **alucinação**. Daí a regra que o RAG operacionaliza: **todo produto destinado à decisão deve ser ancorado em fontes e conferido contra elas**.

> **📝 Nota** — No livro, a sumarização usa o `facebook/bart-large-cnn` (~1,6 GB). Aqui empregamos um modelo dedicado **compacto** (`Falconsai/text_summarization`, ~240 MB, base T5), suficiente para a demonstração — a chamada original fica em comentário. E há um motivo didático extra: na Seção 6.5, pediremos o **mesmo resumo** a um modelo de instrução genérico — e veremos a alucinação acontecer ao vivo.

In [ ]:
# Listagem 6.3 (adaptada) - Sumarizacao abstrativa de material de
# fontes abertas.
from transformers import pipeline

# (No livro: sumarizador = pipeline("summarization",
#                                   model="facebook/bart-large-cnn"))
sumarizador = pipeline("summarization",
                       model="Falconsai/text_summarization")

# texto longo, agregado de multiplas fontes (em ingles: o modelo de
# demonstracao e anglofono; em producao, use um modelo ajustado ao
# portugues -- e valide no idioma real de emprego, Cap. 5)
relatorio = (
    "Open sources reported increased vessel traffic near the northern "
    "coast over the past week. Fishing communities described unusual "
    "movements of unmarked speedboats at night. Port authorities "
    "registered three cargo ships arriving without complete "
    "documentation. A local newspaper reported that customs inspections "
    "were reinforced on Friday. Social media posts from the region "
    "amplified claims of smuggling activity, although no official "
    "confirmation was issued by the authorities.")

resumo = sumarizador(relatorio, max_length=60, min_length=25,
                     do_sample=False)
print(resumo[0]["summary_text"])

# ATENCAO: a sumarizacao abstrativa REESCREVE e pode inserir afirmacoes
# ausentes do original (alucinacao). Todo resumo destinado a uma decisao
# deve ser conferido contra as fontes.

**Observe:** o resumo é útil — e é uma **reescrita seletiva**. Compare-o com o original: ele preservou a ressalva de que *não houve confirmação oficial*? Em material de inteligência, perder (ou inventar) uma ressalva dessas muda a decisão.

---
## 6.5 Riscos: alucinação, vazamento e dependência

Nenhuma tecnologia deste livro exige tanta cautela quanto os LLMs. Três riscos, próprios desses modelos, precisam ser compreendidos e mitigados **antes** de qualquer emprego operacional.

### 6.5.1 Alucinação

Um LLM otimiza **plausibilidade, não verdade**. Ele gera o texto mais provável dado o contexto — e o mais provável pode ser fluente, convincente e **falso**. A **alucinação** — a produção confiante de informação incorreta ou inventada — é *intrínseca* ao modo como o modelo funciona, não um defeito a ser eliminado por completo. Em inteligência, **uma fabricação confiante é pior do que uma lacuna reconhecida**: induz a decisão ao erro sem sinalizar a incerteza. As mitigações são o **RAG** (ancorar em fontes), a exigência de **citações verificáveis** e a **conferência humana**. A regra é categórica: *nunca tratar a saída não ancorada de um LLM como fato*.

### 6.5.2 Vazamento de dados e sigilo

Enviar dados sensíveis ou classificados a um modelo acessado por **API de terceiros** os expõe: os *prompts* podem ser registrados, inspecionados ou usados para treinar versões futuras. Em defesa, isso é uma **falha de segurança operacional** de gravidade evidente — informação que não deveria sair do perímetro sai a cada chamada. A mitigação é estrutural: **modelos abertos e auto-hospedados**, em infraestrutura própria e, quando necessário, em ambientes desconectados — o que conecta este capítulo à IA embarcada e à computação em ambientes negados do Capítulo 9.

### 6.5.3 Dependência de modelos estrangeiros

Os modelos de fronteira são, em sua maioria, desenvolvidos e controlados por atores estrangeiros. Depender deles cria uma **vulnerabilidade estratégica**: o acesso pode ser restringido ou revogado; o modelo pode carregar vieses ou comportamentos não transparentes; e perde-se a **soberania sobre uma capacidade crítica**. É a mesma lógica de soberania tecnológica do Capítulo 1, agora no ponto mais agudo. A resposta — coerente com a Estratégia Nacional de Defesa — passa por **priorizar modelos soberanos ou abertos**, ajustados e operados sob controle nacional, e por construir capacidade própria de desenvolvimento e avaliação.

> **⚠️ Armadilha comum** — **Tratar um LLM como uma fonte de verdade.** O modelo é um gerador de texto plausível, não uma base de fatos nem um oráculo. Usá-lo sem ancoragem em fontes (alucinação), enviar-lhe dados sensíveis (vazamento) ou construir uma capacidade crítica sobre um serviço estrangeiro (dependência) são **três formas do mesmo erro: confundir fluência com confiabilidade**. A saída de um LLM é um rascunho a verificar, jamais um veredito — e, em decisões de defesa, a verificação humana contra fontes é inegociável.

In [ ]:
# Alucinacao ao vivo: o mesmo resumo, por um modelo de instrucao
# generico (o gerador que a Secao 6.6 usara no RAG)
gerador = pipeline("text2text-generation", model="google/flan-t5-base")

resumo_llm = gerador("summarize: " + relatorio, max_length=60)
print("resumo do modelo de instrucao generico:")
print(" ", resumo_llm[0]["generated_text"])

# Agora procure, no relatorio original, o LOCAL que o modelo citou.
# Na nossa execucao de referencia, ele situou os eventos em uma ilha
# grega que NAO consta do relatorio -- fluente, especifico e falso.
print("\n'coast' esta no original?", "coast" in relatorio)
print("O local citado pelo modelo esta no original? Verifique voce mesmo.")

**Observe:** o modelo de instrução genérico produziu um resumo **fluente, específico — e fabricado**: na execução de referência, ele situou os eventos em um local que **não existe no relatório**. Nenhum aviso, nenhuma incerteza sinalizada. É a alucinação da Seção 6.5.1 acontecendo diante de você — e a razão pela qual a saída não ancorada de um LLM **jamais** entra em um produto de decisão. A resposta de engenharia é o padrão da próxima seção: **ancorar a geração em fontes recuperadas**.

---
## 6.6 Aplicação integrada: assistente de análise OSINT com RAG

A aplicação que fecha o Módulo II reúne tudo: um **assistente de análise OSINT** que responde a perguntas do analista **a partir de uma base de documentos-fonte, citando-as**. O pipeline indexa os documentos por *embeddings* de recuperação (Capítulo 5), recupera as passagens mais relevantes a cada pergunta, e pede a um modelo gerador que responda **usando apenas esse contexto** e cite as fontes.

Primeiro, a base de documentos — relatos sintéticos de fontes abertas (em inglês, o idioma do modelo gerador de demonstração; a arquitetura é idêntica para qualquer idioma):

In [ ]:
# Base de documentos-fonte do assistente (sintetica, estilo OSINT)
documentos = [
    "Port authority bulletin: three cargo ships arrived at the northern "
    "port on Monday without complete customs documentation.",
    "Local newspaper: customs inspections at the northern port were "
    "reinforced on Friday after repeated documentation irregularities.",
    "Fishermen association note: unmarked speedboats were seen moving "
    "at night near the northern coast during the past week.",
    "Regional news: a new radar station for coastal surveillance began "
    "operating in the eastern sector last month.",
    "Public tender record: the navy opened bids for the maintenance of "
    "two patrol vessels, with delivery expected next year.",
    "Weather service: strong winds and reduced visibility are expected "
    "along the coast during the next three days.",
    "City council minutes: the fishing community asked for improved "
    "lighting and security at the small docks of the northern bay.",
    "Trade gazette: fuel imports through the northern port grew twelve "
    "percent in the last quarter, according to official figures.",
]
print(f"{len(documentos)} documentos-fonte indexaveis.")

In [ ]:
# Listagem 6.4 - Assistente de analise OSINT com geracao ancorada em
# recuperacao (RAG).
from sentence_transformers import SentenceTransformer
from transformers import pipeline

# Assistente OSINT com RAG: responde a partir de fontes recuperadas e
# as CITA -- reduzindo a alucinacao e tornando a resposta auditavel.

# 1. Indexa os documentos-fonte por embeddings de recuperacao.
codificador = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
indice = codificador.encode(documentos, normalize_embeddings=True)

def recuperar(consulta, k=4):
    q = codificador.encode([consulta], normalize_embeddings=True)[0]
    sim = indice @ q          # cosseno (vetores normalizados)
    return [int(i) for i in np.argsort(sim)[::-1][:k]]

# 2. Monta o contexto com as passagens recuperadas e gera a resposta
#    (reusa o gerador flan-t5 ja carregado na Secao 6.5).

def responder(consulta):
    ids = recuperar(consulta)
    contexto = "\n".join(f"[{i}] {documentos[i]}" for i in ids)
    prompt = (f"Context:\n{contexto}\n\n"
              "Using only the context, answer the question in a "
              "complete sentence and cite the source numbers in "
              "brackets. If the context does not contain the answer, "
              "reply exactly: no basis in the sources.\n\n"
              f"Question: {consulta}\nAnswer:")
    resposta = gerador(prompt, max_length=256)[0]["generated_text"]
    return resposta, ids

# A resposta cita [i] as fontes; o analista as CONFERE antes de usar.
# Modelo auto-hospedado preserva sigilo e soberania (Cap. 9 e 10).
print("Assistente RAG pronto.")

In [ ]:
# O assistente em acao: perguntas COM e SEM base nos documentos
perguntas = [
    "What happened at the northern port regarding documentation?",
    "What was seen near the northern coast at night?",
    "How many submarines does the neighboring country operate?",   # SEM base
]
for p in perguntas:
    resposta, ids = responder(p)
    print(f"PERGUNTA:  {p}")
    print(f"RESPOSTA:  {resposta}")
    print(f"FONTES RECUPERADAS: {ids}\n")

**Observe:** nas duas primeiras perguntas, a resposta vem **das fontes** — e os índices recuperados permitem ao analista **conferir o original** antes de usar. Na terceira, não há base nos documentos, e o assistente **nega — em vez de inventar** (por vezes de forma lacônica; o Exercício 6.3 explora esse caso com a fórmula combinada). Note também a imperfeição honesta: o modelo compacto **nem sempre cita** os números das fontes como instruído — modelos pequenos seguem instruções de forma imperfeita, e é por isso que a lista de fontes recuperadas é impressa *programaticamente*, fora do alcance do gerador.

Esse arranjo é a síntese da postura defendida em todo o módulo. O **RAG combate a alucinação** ancorando a resposta em fontes; a **citação a torna auditável**; o **modelo auto-hospedado preserva sigilo e soberania**; e o **analista, ao conferir as fontes citadas antes de agir, mantém o controle humano significativo** sobre a decisão. O assistente não decide — **sintetiza e evidencia**, para que um humano decida melhor e mais rápido. É esse o emprego responsável de um LLM em inteligência, e é a ele que a governança do Capítulo 10 dará forma normativa.

> **✏️ Experimente:** aumente `k` em `recuperar` (de 4 para 8) e refaça as perguntas. Mais contexto ajuda ou dilui? Em RAG, o número de passagens recuperadas é um hiperparâmetro operacional — pouco contexto perde a resposta; demais, afoga o modelo em irrelevância.

---
## 6.7 O caminho à frente

Com o *Transformer*, as duas famílias de LLMs, os paradigmas de adaptação e o padrão RAG, **encerra-se o Módulo II** — e com ele a percepção: a extração de sentido, visual e textual, do fluxo bruto de dados. O que se percebe, porém, ainda não é o que se decide.

O **Módulo III** trata da **decisão autônoma**. O **Capítulo 7** aborda o aprendizado por reforço profundo — como um agente aprende a agir por tentativa e recompensa, dos processos de decisão de Markov ao *Deep Q-Network* e aos métodos de política —, com aplicações em decisão tática e controle de plataformas. O **Capítulo 8** confronta a **robustez adversarial**: as vulnerabilidades dos modelos — inclusive dos LLMs deste capítulo — a ataques deliberados, e as defesas correspondentes.

A disciplina segue, agora sob apostas maiores: da percepção que *informa* passamos à decisão que *age*. Comparar contra uma *baseline*, medir pelo custo do erro, validar no emprego real e manter o humano no centro deixam de ser metodologia de projeto para se tornarem — quando um modelo passa a agir — **requisitos de segurança**.

## 📋 Resumo do capítulo

- O **Transformer** substitui a recorrência pela **auto-atenção**: cada token atende a todos os demais, via consultas, chaves e valores, na forma $\mathrm{softmax}(QK^\top/\sqrt{d_k})\,V$, produzindo **representações contextuais**. Múltiplas cabeças, codificação posicional e conexões residuais completam a pilha.

- Sua **paralelização** permitiu escalá-lo, e da escala emergiram capacidades não programadas — poderosas e **difíceis de auditar**.

- Duas famílias especializam o Transformer: **BERT** (codificador, bidirecional) para **compreensão** — classificação, extração — e **GPT** (decodificador, autorregressivo) para **geração** — síntese, redação.

- A adaptação admite ***fine-tuning*** (ajusta pesos; controle e soberania, a custo), ***prompt engineering*** (modelo congelado; flexível, mas expõe dados em APIs) e **RAG** (ancora a resposta em fontes recuperadas; reduz alucinação e adiciona auditabilidade).

- Em **OSINT**, os LLMs sintetizam — extraem, resumem, redigem relatórios —, transformando o analista de **leitor em revisor**; mas todo produto para decisão deve ser **ancorado e conferido contra as fontes**.

- Os três riscos de primeira ordem — **alucinação** (plausibilidade não é verdade), **vazamento** (dados a terceiros) e **dependência de modelos estrangeiros** — mitigam-se com RAG e citações, modelos abertos e auto-hospedados, e **controle humano** sobre a decisão.

## ⚠️ Armadilhas comuns

1. **Tratar a saída de um LLM como fato.** O modelo gera texto plausível, não verdade. Ancore em fontes (RAG), exija citações verificáveis e confira contra o original antes de qualquer uso em decisão.

2. **Enviar dados sensíveis a uma API estrangeira.** Cada chamada pode expor informação que não deveria sair do perímetro. Prefira modelos abertos e auto-hospedados, em infraestrutura própria.

3. **Construir capacidade crítica sobre um modelo estrangeiro.** O acesso pode ser revogado e a soberania, perdida. Priorize modelos soberanos ou abertos, ajustados e operados sob controle nacional.

4. **Usar um modelo gerador onde bastava um classificador.** Para classificar ou extrair, um BERT ajustado (ou mesmo a *baseline* TF-IDF do Capítulo 5) costuma ser mais barato, mais rápido e mais auditável do que um grande modelo gerador.

5. **Confiar na sumarização abstrativa sem conferência.** Ela reescreve o texto e pode inserir afirmações ausentes do original. Verifique todo resumo destinado à decisão contra as fontes.

6. **Delegar a decisão ao modelo.** O comportamento emergente dos LLMs é difícil de prever e auditar. A saída é um rascunho a verificar; a decisão permanece humana, com controle humano significativo (Capítulo 10).

---
## 🧭 Exercícios

Classificação: **Essencial** (fixação) · **Tático** (aplicação) · **Estratégico** (extensão criativa). Os exercícios de código têm células preparadas abaixo; os dissertativos podem ser respondidos em células de texto no próprio notebook.

### Essencial

**Exercício 6.1** — Execute a célula abaixo (Listagem 6.1 com matriz de entrada pequena e projeções aleatórias). Verifique que **cada linha da matriz de pesos soma 1** e explique, em duas linhas, o papel do fator $1/\sqrt{d_k}$ na estabilização dos escores antes do *softmax*.

In [ ]:
# Exercicio 6.1 - pesos que somam 1 e o papel do fator 1/sqrt(d_k)
rng = np.random.default_rng(SEMENTE)

n_tokens, d_modelo, d_k = 6, 32, 64   # <--- ALTERE d_k: 4, 64, 256
X = rng.normal(0, 1, (n_tokens, d_modelo))
escala_w = 1 / np.sqrt(d_modelo)      # inicializacao sensata (Cap. 3)
Wq = rng.normal(0, escala_w, (d_modelo, d_k))
Wk = rng.normal(0, escala_w, (d_modelo, d_k))
Wv = rng.normal(0, escala_w, (d_modelo, d_k))

saida, pesos = auto_atencao(X, Wq, Wk, Wv)
print("soma de cada linha da matriz de pesos:", pesos.sum(axis=1).round(6))

# O que acontece SEM o fator de escala?
Q, K = X @ Wq, X @ Wk
sem_escala = softmax(Q @ K.T)                     # escores crus
com_escala = softmax(Q @ K.T / np.sqrt(d_k))      # escores escalonados
print(f"\nmaior peso por linha SEM escala: {sem_escala.max(axis=1).round(3)}")
print(f"maior peso por linha COM escala: {com_escala.max(axis=1).round(3)}")

# Sua resposta (2 linhas):
# Sem o fator, a variancia dos escores cresce com d_k e o softmax
# satura em ... ; com ele, os pesos ...

**Exercício 6.2** — Use a Listagem 6.2 para obter os vetores da palavra *manga* nas duas frases dadas e calcule a **similaridade de cosseno** entre eles. Compare-a à similaridade entre **dois usos da palavra no mesmo sentido** e explique, em uma frase, o que isso demonstra sobre *embeddings* contextuais. *(Requer o BERT da Seção 6.2 carregado.)*

In [ ]:
# Exercicio 6.2 - mesma acepcao vs. acepcoes diferentes
frases = {
    "fruta 1": "comi uma manga madura",
    "fruta 2": "colhi uma manga madura do pe",        # <--- ALTERE as
    "roupa":   "rasguei a manga da camisa",           # frases se quiser
}
vetores_manga = {}
for nome_frase, f in frases.items():
    v, toks = embeddings_contextuais(f)
    vetores_manga[nome_frase] = vetor_do_token(v, toks)

print("similaridade de cosseno entre ocorrencias de 'manga':")
print(f"  fruta 1 ~ fruta 2 (MESMA acepcao):      "
      f"{cosseno(vetores_manga['fruta 1'], vetores_manga['fruta 2']):.3f}")
print(f"  fruta 1 ~ roupa  (acepcoes DIFERENTES): "
      f"{cosseno(vetores_manga['fruta 1'], vetores_manga['roupa']):.3f}")

# Sua resposta (1 frase):
# As embeddings contextuais demonstram que ...

**Exercício 6.3** — Na célula abaixo (assistente da Listagem 6.4), formule uma pergunta **cuja resposta não esteja** nos documentos indexados e verifique se o assistente **reconhece a ausência de base** em vez de inventar. Em duas linhas, relacione o resultado ao papel do RAG no combate à alucinação. *(Requer o assistente da Seção 6.6 carregado.)*

In [ ]:
# Exercicio 6.3 - o assistente reconhece o que NAO sabe?
minha_pergunta = ("What is the range of the new coastal missile "
                  "battery?")            # <--- ALTERE: pergunte algo
                                         # que NAO esta nos documentos
resposta, ids = responder(minha_pergunta)
print(f"PERGUNTA:  {minha_pergunta}")
print(f"RESPOSTA:  {resposta}")
print(f"FONTES RECUPERADAS: {ids}")

# Sua resposta (2 linhas):
# 1) O assistente reconheceu a ausencia de base ou tentou inventar?
# 2) O RAG combate a alucinacao porque ... (e onde ele ainda falha?)

### Tático

**Exercício 6.4** *(dissertativo)* — Compare, em uma **tabela**, *fine-tuning*, *prompt engineering* e RAG segundo quatro critérios: **custo de preparação**, **controle sobre o modelo**, **exposição de dados** e **facilidade de atualizar o conhecimento**. Recomende, justificando, a estratégia adequada para (a) classificar milhares de relatórios por tema e (b) responder perguntas sobre um acervo documental que muda semanalmente.

*Responda em uma célula de texto abaixo.*

**Exercício 6.5** — A célula abaixo estende a Listagem 6.4 para que a resposta liste explicitamente os **identificadores das fontes usadas e o trecho de cada uma**. Execute-a e discuta, em poucas linhas, como essa **rastreabilidade** apoia a verificação humana e a auditoria da decisão.

In [ ]:
# Exercicio 6.5 - resposta com rastreabilidade explicita das fontes
def responder_rastreavel(consulta, k=4):
    ids = recuperar(consulta, k=k)
    contexto = "\n".join(f"[{i}] {documentos[i]}" for i in ids)
    prompt = ("Answer the question USING ONLY the context below and "
              "cite the sources in brackets.\n\n"
              f"Context:\n{contexto}\n\nQuestion: {consulta}\nAnswer:")
    resposta = gerador(prompt, max_length=256)[0]["generated_text"]
    print(f"PERGUNTA: {consulta}")
    print(f"RESPOSTA: {resposta}\n")
    print("FONTES USADAS (para conferencia humana):")
    for i in ids:
        print(f"  [{i}] {documentos[i][:76]}...")
    return resposta, ids

responder_rastreavel("What happened at the northern port regarding "
                     "documentation?")

# Sua resposta (poucas linhas):
# A rastreabilidade apoia a verificacao humana porque ...
# E apoia a auditoria da decisao porque ...

**Exercício 6.6** *(dissertativo)* — Escolha uma tarefa de **compreensão** (por exemplo, a classificação temática do Capítulo 5) e uma de **geração** (por exemplo, sumarização) e indique, para cada uma, qual família de modelo — **BERT ou GPT** — é mais adequada e por quê. Estime, em termos qualitativos, o custo de implantação de cada escolha.

*Responda em uma célula de texto abaixo.*

### Estratégico

**Exercício 6.7** — Em um texto técnico (até três páginas), analise os três riscos de emprego de LLMs em defesa — **alucinação**, **vazamento de dados** e **dependência de modelos estrangeiros** — e proponha, para cada um, medidas concretas de mitigação em nível **técnico**, de **processo** e de **política institucional**. Relacione o argumento à soberania tecnológica do Capítulo 1 e à Estratégia Nacional de Defesa.

**Exercício 6.8** — Projete, em diagrama, uma **arquitetura de assistente OSINT soberana**, apta a operar em ambiente com restrição de conectividade: especifique o modelo (aberto, auto-hospedado), o mecanismo de recuperação, os pontos de controle humano na decisão e as salvaguardas contra alucinação e vazamento. Justifique cada escolha do ponto de vista de *opsec* e soberania, antecipando as restrições do Capítulo 9.

**Exercício 6.9** — Monte um pequeno acervo de documentos de domínio público e implemente, sobre ele, o assistente RAG da Listagem 6.4. Avalie qualitativamente a **fidelidade das respostas às fontes**, documente em um miniartigo de até três páginas os casos de alucinação observados e proponha ajustes (no *prompt*, no número de passagens recuperadas ou no modelo) que os reduzam — discutindo os limites de generalização para um acervo operacional real.

---

*Fim do Capítulo 6 — encerramos o Módulo II. No Capítulo 7, a decisão autônoma: o aprendizado por reforço profundo.*